# Instancia Premier League 2021-2022

Descarga desde Football-Data la base de esta temporada y la usa como conjunto de equipos para resolver un calendario doble Round-Robin nuevo con el modelo compartido `modelo_calendario.py`.

## Qué representa esta instancia

El CSV histórico de 2021-2022 se usa para obtener y validar los 20 clubes de esa temporada. No se busca reproducir el calendario oficial: el modelo genera una programación nueva de 38 jornadas y 380 partidos, con cada pareja jugando dos veces, una en cada estadio.

## Flujo del notebook

1. Localiza el proyecto e importa `resolver_instancia` desde `modelo_calendario.py`.
2. Descarga los partidos históricos desde Football-Data y toma de ellos la lista de equipos.
3. Resuelve la instancia dos veces: con parámetros predeterminados de Gurobi y con cortes y heurísticas desactivados.
4. Compara estado, objetivo, cota, gap, tiempo y nodos; luego muestra el calendario nuevo y una muestra de los datos de origen.

Las listas de clásicos, estadios compartidos, equipos europeos y jornadas críticas están vacías; esas restricciones no aportan efecto hasta completar los datos. El término `viajes` es una aproximación basada en jugar de visita en una jornada crítica, no una distancia real.

## Licencia para ejecutar

Antes de ejecutar la celda Python que resuelve el modelo, activa la licencia desde PowerShell o la terminal de Windows con el comando que entrega Gurobi: `grbgetkey CLAVE_ENTREGADA_POR_GUROBI` (reemplaza el texto por la clave real, sin escribirla en el notebook). Después reinicia el kernel y ejecuta las celdas. Si `grbgetkey` no se reconoce, instala Gurobi Optimizer para Windows o agrega su carpeta `bin` al `PATH`. La lectura del CSV por sí sola no requiere licencia.

In [2]:
# Evaluar la instancia 2021-2022 con GurobiPy en dos configuraciones.
from pathlib import Path
import sys

import pandas as pd
import time
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))
from modelo_calendario import resolver_instancia

TEMPORADA = "2021-2022"
URL_CSV = str((Path.cwd().resolve().parent / "BASES" / "2021-2022.csv"))
PARAMETROS_MODELO = {
    "clasicos": [],
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "numero_equipos_esperado": 20,
}

configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}

TIEMPO_TOTAL = 3600
MARGEN_SEGUNDOS = 30
inicio = time.monotonic()
resultados = {}

for indice, (nombre, desactivar_cortes_heuristicas) in enumerate(configuraciones.items()):
    pendientes = len(configuraciones) - indice
    disponible = TIEMPO_TOTAL - (time.monotonic() - inicio) - MARGEN_SEGUNDOS

    if disponible < 1:
        print(f"Se omite {nombre}: se agotó el presupuesto de tiempo.")
        break

    tiempo_limite = max(1, int(disponible / pendientes))
    print(f"{nombre}: límite de Gurobi de {tiempo_limite} segundos.")

    resultados[nombre] = resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        tiempo_limite=tiempo_limite,
        **PARAMETROS_MODELO,
    )

reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
reporte["Objetivo (ObjVal)"] = reporte["Objetivo (ObjVal)"].fillna("No factible")
reporte["Gap (MIPGap)"] = reporte["Gap (MIPGap)"].fillna("No factible")
estado_map = {
    1: "OPTIMAL",
    2: "INFEASIBLE",
    3: "INF_OR_UNBD",
    4: "CUTOFF",
    5: "LOADED",
    6: "UNKNOWN",
}
reporte["Estado Gurobi"] = reporte["Estado Gurobi"].map(lambda v: estado_map.get(v, v))
display(reporte)

resultado_normal = resultados["Normal (Gurobi por defecto)"]
print(f"Partidos históricos descargados: {len(resultado_normal['partidos_historicos'])}")
print(f"Breaks: {resultado_normal['breaks']} | Viajes críticos: {resultado_normal['viajes_criticos']}")
display(resultado_normal["calendario"])
display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))

Set parameter TimeLimit to value 3600
Set parameter OutputFlag to value 1


2021-2022: 20 equipos, 38 jornadas, 16,680 variables y 7,080 restricciones.
Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: AMD Ryzen 7 5700U with Radeon Graphics, instruction set [SSE2|AVX|AVX2]
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  3600

Optimize a model with 7080 rows, 16680 columns and 71800 nonzeros (Min)
Model fingerprint: 0x1921d8ff
Model has 1480 linear objective coefficients
Variable types: 1480 continuous, 15200 integer (15200 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 2e+01]

Presolve removed 2960 rows and 0 columns
Presolve time: 0.38s
Presolved: 4120 rows, 16680 columns, 65880 nonzeros
Variable types: 0 continuous, 16680 integer (16680 binary)

Root simplex log...

Iteration    Objective       Primal Inf.    Dual Inf.      Time
    6034

,Configuración,Estado Gurobi,Objetivo (ObjVal),Cota (ObjBoundC),Gap (MIPGap),Tiempo (s),Nodos (NodeCount)
0,Normal (Gurobi por defecto),9,42.0,5.142857,0.857143,3600.086,1813.0
1,Sin cortes ni heurísticas,9,NaN,5.142857,NaN,8392.680,881.0


Partidos históricos descargados: 380
Breaks: 42.0 | Viajes críticos: 0


,Jornada,Local,Visitante
0,1,Arsenal,Aston Villa
1,1,Brentford,Brighton
2,1,Burnley,Chelsea
3,1,Crystal Palace,Everton
4,1,Leeds,Leicester
...,...,...,...
375,38,Liverpool,Burnley
376,38,Man City,Brentford
377,38,Southampton,Tottenham
378,38,Watford,Newcastle


,Date,HomeTeam,AwayTeam
0,13/08/2021,Brentford,Arsenal
1,14/08/2021,Man United,Leeds
2,14/08/2021,Burnley,Brighton
3,14/08/2021,Chelsea,Crystal Palace
4,14/08/2021,Everton,Southampton
5,14/08/2021,Leicester,Wolves
6,14/08/2021,Watford,Aston Villa
7,14/08/2021,Norwich,Liverpool
8,15/08/2021,Newcastle,West Ham
9,15/08/2021,Tottenham,Man City
